# Plot basic session figures

This notebook shows how to make a few useful figures from one packed h5 session: field of view, example neural traces, trial-type fractions, and interval distributions.

## 1. Setup

Change `H5_PATH` to any packed session under `results_pack/<subject>/<session>/<session>.h5`.

In [ ]:
from pathlib import Path
import os

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)

H5_PATH = repo / 'results_pack' / 'YH21SC' / 'V1_random_01' / 'V1_random_01.h5'
print(H5_PATH)
print('exists:', H5_PATH.exists())

## 2. Load the arrays used for plotting

The project plotters usually receive `list_labels`, `list_masks`, and `list_neural_trials`. For one session, each list has one item.

In [ ]:
import h5py
import numpy as np

with h5py.File(H5_PATH, 'r') as f:
    labels = f['labels'][:]
    masks = f['masks']['masks_func'][:]
    mean_func = f['masks']['mean_func'][:]
    max_func = f['masks']['max_func'][:]
    mean_anat = f['masks']['mean_anat'][:] if 'mean_anat' in f['masks'] else None
    masks_anat = f['masks']['masks_anat'][:] if 'masks_anat' in f['masks'] else None
    neural_trials = {key: f['neural_trials'][key][:] for key in f['neural_trials'].keys()}

list_labels = [labels]
list_neural_trials = [neural_trials]

print('labels:', labels.shape)
print('masks:', masks.shape)
print('mean_func:', mean_func.shape)
print('max_func:', max_func.shape)
print('dff:', neural_trials['dff'].shape)
print('stim_labels:', neural_trials['stim_labels'].shape)

## 3. Field-of-view figure

This uses the same mask plotting class as the report. The first panel overlays ROI boundaries on the functional image; the second panel shows the labeled ROI map.

In [ ]:
import matplotlib.pyplot as plt
from plot.fig1_mask import plotter_main_masks

plotter = plotter_main_masks(labels, masks, mean_func, max_func, mean_anat, masks_anat)

fig, axs = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
plotter.plot_func(axs[0], 'mean')
plotter.plot_func_masks_color(axs[1])
plotter.plot_func(axs[2], 'max', with_mask=False)
plt.show()

## 4. Pick one ROI and show its location

Change `ROI_ID` to inspect another neuron. ROI IDs are zero-based in Python.

In [ ]:
ROI_ID = 0

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), constrained_layout=True)
plotter.roi_loc_1chan(axs[0], ROI_ID, 'mean')
plotter.roi_func(axs[1], ROI_ID, 'mean')
plt.show()

## 5. Example raw traces by cell label

This calls the same helper used in the field-of-view report page. It randomly samples up to 50 traces per cell label.

In [ ]:
from plot.fig2_raw_traces import plot_sess_example_traces

label_names = {'-1': 'Exc', '1': 'VIP', '2': 'SST'}

fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)
plot_sess_example_traces(ax, list_labels, list_neural_trials, label_names)
plt.show()

## 6. Trial composition figure

`stim_labels` contains one row per visual stimulus. These panels summarize stimulus identity and task structure.

In [ ]:
from plot.fig3_intervals import plot_stim_type, plot_random_type, plot_fix_jitter_type, plot_standard_type

fig, axs = plt.subplots(1, 4, figsize=(14, 3.5), constrained_layout=True)
plot_stim_type(axs[0], list_neural_trials)
plot_random_type(axs[1], list_neural_trials)
plot_fix_jitter_type(axs[2], list_neural_trials)
plot_standard_type(axs[3], list_neural_trials)
plt.show()

## 7. Interval distribution figure

The interval is measured from one stimulus offset to the next stimulus onset: `stim_labels[1:, 0] - stim_labels[:-1, 1]`.

In [ ]:
from plot.fig3_intervals import plot_random_isi_distribution, plot_jitter_isi_distribution, plot_stim_label, plot_trial_legend

fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
plot_random_isi_distribution(axs[0, 0], list_neural_trials)
plot_jitter_isi_distribution(axs[0, 1], list_neural_trials)
plot_stim_label(axs[1, 0], list_neural_trials, [0, 3000])
plot_trial_legend(axs[1, 1])
plt.show()

## 8. Save an example figure

Set `SAVE_FIG = True` to write an SVG into `results/notebook_examples`.

In [ ]:
SAVE_FIG = False

if SAVE_FIG:
    out_dir = repo / 'results' / 'notebook_examples'
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(1, 1, figsize=(5, 4), constrained_layout=True)
    plotter.plot_func(ax, 'mean')
    fig.savefig(out_dir / 'example_fov.svg')
    print(out_dir / 'example_fov.svg')
else:
    print('Skipped saving.')